# Intelli-Speech - MAINFILE (existing-artifact workflow)

This notebook is the **single active working notebook**. It reads artifacts that already
exist in Google Drive. It does **not** generate datasets, install packages, train models,
or run inference.

## What this notebook does

1. Confirms that the runtime is already recovered.
2. Inventories the existing `paired_dataset_v3` metadata and audio.
3. Verifies the recorded dataset provenance (manifest fingerprint and report files).
4. Restores the existing STFT checkpoint and re-validates the round trip.
5. Inventories existing experiment outputs.

## Where the other responsibilities live

| Concern | Location |
|---|---|
| Runtime, cache, Hugging Face, checkpoint, ClearerVoice recovery | `RECOVERY.ipynb` |
| Verified progress, blockers, decisions, next task | `PROJECT_NOTES.md` |
| Historical experiments (v1/v2/v3 generation, IRM, Wiener, FRCRN drafts) | `prev_files/MAINFILE_monolithic_2026-11-21.ipynb` |
| Dataset structure and audit record | `docs/dataset_structure.md` |

## Execution order

Run `RECOVERY.ipynb` first in a fresh Colab runtime, then run this notebook top to bottom.

## Safety rules

- Read-only. This notebook performs no writes to Drive.
- Do not regenerate the paired dataset from this notebook.
- Do not overwrite files in `Intelli-Speech-data/experiments/`.
- Keep original datasets, legacy notebooks, and historical outputs unchanged.


In [ ]:
# ============================================================
# STEP 0 | CONFIGURATION ONLY
# No file access, no installs, no writes.
# ============================================================
from pathlib import Path

DRIVE_ROOT = Path("/content/drive")
REPO_DIR = Path("/content/Intelli-Speech")
PROJECT_DIR = DRIVE_ROOT / "MyDrive" / "Intelli-Speech-data"

DATASET_DIR = PROJECT_DIR / "paired_dataset_v3"
METADATA_PATH = DATASET_DIR / "metadata.csv"
CLEAN_DIR = DATASET_DIR / "clean"
NOISY_DIR = DATASET_DIR / "noisy"
MANIFEST_PATH = (
    PROJECT_DIR / "paired_dataset_v3_manifest_speaker_balanced.csv"
)
STFT_CHECKPOINT_PATH = PROJECT_DIR / "analysis_cache" / "stft_example.npz"
EXPERIMENT_ROOT = PROJECT_DIR / "experiments"

# Recorded, independently audited expectations (see docs/dataset_structure.md).
EXPECTED_CLEAN_FILES = 500
EXPECTED_NOISY_FILES = 1000
EXPECTED_METADATA_ROWS = 1000
EXPECTED_SAMPLE_RATE = 16000
EXPECTED_MANIFEST_SHA256 = (
    "2af46ef5c9a4545c0775f08a07b734bc0696a888339a18d5e40cdd1103941eb0"
)

print("Configuration loaded. No files were read or written.")
print("Dataset:", DATASET_DIR)
print("STFT checkpoint:", STFT_CHECKPOINT_PATH)


In [ ]:
# ============================================================
# STEP 1 | PRECONDITIONS
# This notebook never mounts Drive and never installs packages.
# If a check fails, run RECOVERY.ipynb first.
# ============================================================
import importlib.util
import sys

required_modules = ("numpy", "pandas", "soundfile", "librosa")
missing_modules = [
    name
    for name in required_modules
    if importlib.util.find_spec(name) is None
]
if missing_modules:
    raise RuntimeError(
        "Missing Python modules: "
        + ", ".join(missing_modules)
        + ". Run RECOVERY.ipynb (steps 1-3) to restore the runtime. "
        "This notebook does not install packages."
    )

if not DRIVE_ROOT.is_dir():
    raise RuntimeError(
        f"Google Drive is not mounted at {DRIVE_ROOT}. "
        "Run RECOVERY.ipynb step 2 first."
    )

if not REPO_DIR.is_dir():
    raise RuntimeError(
        f"Repository not found at {REPO_DIR}. "
        "Run RECOVERY.ipynb step 2 first."
    )

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# The helper package is optional for this notebook: the artifact checks below do not
# depend on it. A warning here means the checked-out commit does not contain
# intelli_speech/, not that the dataset is unusable.
try:
    from intelli_speech import (  # noqa: E402
        AudioLoadError,
        clipping_report,
        compute_snr_db,
        load_audio,
        peak_amplitude,
        resolve_metadata_audio_path,
        rms_amplitude,
        validate_mono_16khz,
    )
    helper_state = "available"
except ImportError as exc:
    helper_state = f"unavailable ({exc})"

print("Preconditions satisfied.")
print("Repository path:", REPO_DIR)
print("Repository helper package:", helper_state)
if helper_state == "available":
    print("Available helpers:", ", ".join(sorted((
        "AudioLoadError", "clipping_report", "compute_snr_db", "load_audio",
        "peak_amplitude", "resolve_metadata_audio_path", "rms_amplitude",
        "validate_mono_16khz",
    ))))
else:
    print(
        "Warning: intelli_speech/ is not present in this checkout. The artifact "
        "checks continue; commit the helper package when it is ready."
    )


In [ ]:
# ============================================================
# STEP 2 | EXISTING DATASET INVENTORY (read-only)
# Reads metadata and counts files. Does not read or modify audio.
# ============================================================
from pathlib import Path

import pandas as pd

if not METADATA_PATH.is_file():
    raise FileNotFoundError(f"Dataset metadata not found: {METADATA_PATH}")

metadata = pd.read_csv(METADATA_PATH)

clean_files = sorted(CLEAN_DIR.glob("*.wav")) if CLEAN_DIR.is_dir() else []
noisy_files = sorted(NOISY_DIR.glob("*.wav")) if NOISY_DIR.is_dir() else []

print("=== EXISTING DATASET INVENTORY ===")
print("Dataset directory:", DATASET_DIR)
print("Metadata rows:", len(metadata))
print("Metadata columns:", metadata.columns.tolist())
print("Clean WAV files:", len(clean_files))
print("Noisy WAV files:", len(noisy_files))

# Known columns from the verified v3 pipeline.
for column in ("clean_file", "noisy_file"):
    print(f"  contains {column!r}: {column in metadata.columns}")

for column in ("split", "target_snr_db", "noise_source", "noise_category", "gender"):
    if column in metadata.columns:
        counts = metadata[column].value_counts(dropna=False)
        print(f"\n{column} distribution:")
        for value, count in counts.items():
            print(f"  {value}: {count}")

checks = {
    "clean file count matches the audited 500":
        len(clean_files) == EXPECTED_CLEAN_FILES,
    "noisy file count matches the audited 1000":
        len(noisy_files) == EXPECTED_NOISY_FILES,
    "metadata row count matches the audited 1000":
        len(metadata) == EXPECTED_METADATA_ROWS,
    "clean_file column present": "clean_file" in metadata.columns,
    "noisy_file column present": "noisy_file" in metadata.columns,
}
print("\nInventory checks:")
for label, passed in checks.items():
    print(f"  [{'PASS' if passed else 'REVIEW'}] {label}")

if not all(checks.values()):
    print(
        "\nNote: a REVIEW result means the artifact state differs from the audited "
        "record. Do not regenerate data; report the difference instead."
    )


In [ ]:
# ============================================================
# STEP 3 | DATASET PROVENANCE (read-only)
# Verifies the recorded manifest fingerprint and generation artifacts.
# ============================================================
import hashlib
import json
from pathlib import Path

def sha256_of(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

print("=== DATASET PROVENANCE ===")

if MANIFEST_PATH.is_file():
    manifest_sha256 = sha256_of(MANIFEST_PATH)
    print("Manifest:", MANIFEST_PATH.name)
    print("Recorded SHA-256:", EXPECTED_MANIFEST_SHA256)
    print("Current SHA-256: ", manifest_sha256)
    print("Fingerprint match:", manifest_sha256 == EXPECTED_MANIFEST_SHA256)
    if manifest_sha256 != EXPECTED_MANIFEST_SHA256:
        print(
            "The manifest has changed since the audited build. "
            "Do not rebuild the dataset; report the difference first."
        )
else:
    print("Manifest not found:", MANIFEST_PATH)

for name in ("build_config.json", "dataset_report.json"):
    path = DATASET_DIR / name
    if not path.is_file():
        print(f"{name}: missing")
        continue
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    print(f"{name}: present, type={type(payload).__name__}")
    if isinstance(payload, dict):
        print(f"  {len(payload)} top-level key(s):", sorted(payload)[:15])
    elif isinstance(payload, list):
        print(f"  {len(payload)} item(s)")

progress_path = DATASET_DIR / "progress.jsonl"
if progress_path.is_file():
    with progress_path.open("r", encoding="utf-8") as handle:
        line_count = sum(1 for line in handle if line.strip())
    print(f"progress.jsonl: {line_count} record(s)")
else:
    print("progress.jsonl: missing")


In [ ]:
# ============================================================
# STEP 4 | EXISTING STFT CHECKPOINT (read-only)
# Restores the saved analysis state and re-validates the iSTFT round trip.
# ============================================================
import librosa
import numpy as np

if not STFT_CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f"STFT checkpoint not found: {STFT_CHECKPOINT_PATH}. "
        "Run RECOVERY.ipynb step 5 to inspect checkpoint state. "
        "Do not regenerate it from this notebook."
    )

with np.load(STFT_CHECKPOINT_PATH) as saved:
    clean = saved["clean"].copy()
    noisy = saved["noisy"].copy()
    clean_stft = saved["clean_stft"].copy()
    noisy_stft = saved["noisy_stft"].copy()
    sample_rate = int(saved["sample_rate"])
    n_fft = int(saved["n_fft"])
    hop_length = int(saved["hop_length"])

reconstructed = librosa.istft(
    clean_stft, hop_length=hop_length, n_fft=n_fft, length=len(clean)
)
round_trip_error = float(np.max(np.abs(reconstructed - clean)))

print("=== STFT CHECKPOINT RESTORE ===")
print("Checkpoint:", STFT_CHECKPOINT_PATH.name)
print("Sample rate:", sample_rate)
print("n_fft / hop_length:", n_fft, hop_length)
print("Clean STFT shape:", clean_stft.shape)
print("Noisy STFT shape:", noisy_stft.shape)
print("Waveform length:", len(clean))
print("All STFT values finite:",
      bool(np.isfinite(clean_stft).all() and np.isfinite(noisy_stft).all()))
print("iSTFT round-trip max abs error:", f"{round_trip_error:.3e}")

if sample_rate != EXPECTED_SAMPLE_RATE:
    print("Warning: unexpected sample rate for this project.")
if round_trip_error > 1e-4:
    print("Warning: round-trip error is higher than expected; inspect before reuse.")
else:
    print("Round trip validated for reuse in baseline work.")


In [ ]:
# ============================================================
# STEP 5 | EXISTING EXPERIMENT ARTIFACTS (read-only listing)
# Lists saved experiment outputs. Does not open or overwrite audio.
# ============================================================
print("=== EXPERIMENT ARTIFACTS ===")

if not EXPERIMENT_ROOT.is_dir():
    print("No experiment root found:", EXPERIMENT_ROOT)
else:
    for directory in sorted(
        (item for item in EXPERIMENT_ROOT.iterdir() if item.is_dir()),
        key=lambda item: item.name,
    ):
        files = [item for item in directory.rglob("*") if item.is_file()]
        total_mb = sum(item.stat().st_size for item in files) / (1024 * 1024)
        print(f"{directory.name}: {len(files)} file(s), {total_mb:.2f} MB")
        for item in sorted(files, key=lambda entry: entry.name)[:5]:
            print(f"  {item.name} ({item.stat().st_size / 1024:.1f} KB)")
        if len(files) > 5:
            print(f"  ... {len(files) - 5} more file(s)")
    if not any(item.is_dir() for item in EXPERIMENT_ROOT.iterdir()):
        print("Experiment root is empty. No enhanced audio has been saved yet.")

print("\nNo files were created, modified, or deleted by this notebook.")


## Where to continue

The verified state of the project is recorded in `PROJECT_NOTES.md`. Read it before
starting new work; it lists the blockers and the agreed next task.

Current blockers in short:

1. ClearerVoice could not be installed in the recorded runtime: installing `clearvoice`
   pulled NumPy down to 1.26.4 and `from clearvoice import ClearVoice` then failed.
2. No FRCRN checkpoint is present in any inspected location, so FRCRN has never run.
3. No objective metrics (PESQ/STOI/SNR improvement) have been computed for any baseline.

Rules for the next work:

- Resolve the ClearerVoice environment with an explicit, isolated decision. Do not
  downgrade NumPy inside the main runtime.
- Do not regenerate `paired_dataset_v3`.
- Do not write to `Intelli-Speech-data/experiments/` from a read-only check.
- Record new results in `PROJECT_NOTES.md` only after they are actually observed.
